# World Cup Score Predictor — Clean Feature Engineering Notebook

This notebook builds a clean pre-match feature dataset for the World Cup score predictor.

Main goals:
- One row per match.
- Use only information available before kickoff.
- Convert post-match ELO/rank values into pre-match values.
- Keep a focused, relevant feature set.
- Evaluate with expanding-window World Cup backtesting.

## 1. Imports and Paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from math import exp, factorial

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 120)

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUTS_DIR = BASE_DIR / "outputs" / "evaluation"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)

RAW_DIR: ../data/raw
PROCESSED_DIR: ../data/processed
OUTPUTS_DIR: ../outputs/evaluation


## 2. Load Yearly ELO Match Files

In [ ]:
elo_files = sorted(RAW_DIR.glob("elo_*_results.csv"))

if not elo_files:
    raise FileNotFoundError("No files found matching data/raw/elo_*_results.csv")

dfs = []
for file in elo_files:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    dfs.append(temp)

elo_df = pd.concat(dfs, ignore_index=True)
elo_df["date"] = pd.to_datetime(elo_df["date"], errors="coerce")

before_filter_shape = elo_df.shape

elo_df = elo_df[
    elo_df["date"].dt.year >= 2004
].copy()

elo_df = elo_df.sort_values("date").reset_index(drop=True)

print("Filtered ELO data to 2004+")
print("Before:", before_filter_shape)
print("After:", elo_df.shape)
print("Date range:", elo_df["date"].min().date(), "to", elo_df["date"].max().date())

print("Number of ELO files:", len(elo_files))
print("Loaded ELO shape:", elo_df.shape)
display(elo_df.head())

Filtered ELO data to 2004+
Before: (24260, 16)
After: (21539, 16)
Date range: 2004-01-01 to 2026-05-16
Number of ELO files: 26
Loaded ELO shape: (21539, 16)


,date,team_a,team_b,goals_a,goals_b,competition,location,rating_change_a,rating_change_b,rating_a,rating_b,rank_change_a,rank_change_b,rank_a,rank_b,source_file
0,2004-01-01,Bermuda,Barbados,0,4,Dudley Eve Memorial Trophy,Bermuda,-28,28,1242,1397,-8,7,146,109,elo_2004_results.csv
1,2004-01-01,Kuwait,Yemen,4,0,Gulf Cup,Kuwait,6,-6,1525,1173,2,-1,72,160,elo_2004_results.csv
2,2004-01-01,Saudi Arabia,Bahrain,1,0,Gulf Cup,Kuwait,12,-12,1674,1511,3,-6,44,78,elo_2004_results.csv
3,2004-01-03,Bahrain,Oman,1,0,Gulf Cup,Kuwait,23,-23,1534,1537,7,-4,71,70,elo_2004_results.csv
4,2004-01-03,Qatar,United Arab Emirates,0,0,Gulf Cup,Kuwait,-4,4,1517,1459,-4,1,77,90,elo_2004_results.csv


## 3. Basic Cleaning and No-Leakage Pre-Match Features

In [3]:
df = elo_df.copy()

required_cols = [
    "date", "team_a", "team_b", "goals_a", "goals_b", "competition", "location",
    "rating_change_a", "rating_change_b", "rating_a", "rating_b",
    "rank_change_a", "rank_change_b", "rank_a", "rank_b"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date", "team_a", "team_b", "goals_a", "goals_b"])
df = df.sort_values("date").reset_index(drop=True)

# Targets
df["target_goals_a"] = df["goals_a"].astype(int)
df["target_goals_b"] = df["goals_b"].astype(int)
df["target_goal_diff"] = df["target_goals_a"] - df["target_goals_b"]
df["target_total_goals"] = df["target_goals_a"] + df["target_goals_b"]

# Result points
df["result_a"] = np.where(df["goals_a"] > df["goals_b"], 3,
                  np.where(df["goals_a"] == df["goals_b"], 1, 0))
df["result_b"] = np.where(df["goals_b"] > df["goals_a"], 3,
                  np.where(df["goals_a"] == df["goals_b"], 1, 0))

# Raw rating/rank values are post-match, so reconstruct pre-match values
df["rating_a_before"] = df["rating_a"] - df["rating_change_a"]
df["rating_b_before"] = df["rating_b"] - df["rating_change_b"]

df["rank_a_before"] = df["rank_a"] - df["rank_change_a"]
df["rank_b_before"] = df["rank_b"] - df["rank_change_b"]

df["elo_diff"] = df["rating_a_before"] - df["rating_b_before"]
df["rank_diff"] = df["rank_a_before"] - df["rank_b_before"]

# Home advantage for team_a
df["is_home_adv"] = (
    df["team_a"].astype(str).str.strip().str.lower()
    ==
    df["location"].astype(str).str.strip().str.lower()
).astype(int)

print("Cleaned shape:", df.shape)
display(df[[
    "date", "team_a", "team_b", "competition", "location",
    "rating_a_before", "rating_b_before", "elo_diff",
    "rank_a_before", "rank_b_before", "rank_diff",
    "is_home_adv", "target_goals_a", "target_goals_b"
]].head())

Cleaned shape: (21539, 29)


,date,team_a,team_b,competition,location,rating_a_before,rating_b_before,elo_diff,rank_a_before,rank_b_before,rank_diff,is_home_adv,target_goals_a,target_goals_b
0,2004-01-01,Bermuda,Barbados,Dudley Eve Memorial Trophy,Bermuda,1270,1369,-99,154,102,52,1,0,4
1,2004-01-01,Kuwait,Yemen,Gulf Cup,Kuwait,1519,1179,340,70,161,-91,1,4,0
2,2004-01-01,Saudi Arabia,Bahrain,Gulf Cup,Kuwait,1662,1523,139,41,84,-43,0,1,0
3,2004-01-03,Bahrain,Oman,Gulf Cup,Kuwait,1511,1560,-49,64,74,-10,0,1,0
4,2004-01-03,Qatar,United Arab Emirates,Gulf Cup,Kuwait,1521,1455,66,81,89,-8,0,0,0


## 4. Rolling Team Features

In [4]:
WINDOW = 5
team_history = {}

ELO_BASE = df[["rating_a_before", "rating_b_before"]].stack().mean()

features = {
    "team_a_form_last5": [],
    "team_b_form_last5": [],
    "team_a_goals_for_avg_last5": [],
    "team_b_goals_for_avg_last5": [],
    "team_a_goals_against_avg_last5": [],
    "team_b_goals_against_avg_last5": [],
    "team_a_weighted_goals_for_avg_last5": [],
    "team_b_weighted_goals_for_avg_last5": [],
    "team_a_weighted_goals_against_avg_last5": [],
    "team_b_weighted_goals_against_avg_last5": [],
    "team_a_avg_opponent_elo_last5": [],
    "team_b_avg_opponent_elo_last5": [],
    "team_a_rating_change_avg_last5": [],
    "team_b_rating_change_avg_last5": [],
    "team_a_matches_played_before": [],
    "team_b_matches_played_before": [],
    "team_a_days_since_last_match": [],
    "team_b_days_since_last_match": [],
    "team_a_win_streak": [],
    "team_b_win_streak": [],
}

def mean_or_zero(values):
    return float(np.mean(values)) if len(values) > 0 else 0.0

def current_win_streak(history):
    streak = 0
    for match in reversed(history):
        if match["points"] == 3:
            streak += 1
        else:
            break
    return streak

for _, row in df.iterrows():
    team_a = row["team_a"]
    team_b = row["team_b"]
    date = row["date"]

    team_history.setdefault(team_a, [])
    team_history.setdefault(team_b, [])

    hist_a_all = team_history[team_a]
    hist_b_all = team_history[team_b]

    hist_a = hist_a_all[-WINDOW:]
    hist_b = hist_b_all[-WINDOW:]

    features["team_a_form_last5"].append(mean_or_zero([x["points"] for x in hist_a]))
    features["team_b_form_last5"].append(mean_or_zero([x["points"] for x in hist_b]))

    features["team_a_goals_for_avg_last5"].append(mean_or_zero([x["goals_for"] for x in hist_a]))
    features["team_b_goals_for_avg_last5"].append(mean_or_zero([x["goals_for"] for x in hist_b]))

    features["team_a_goals_against_avg_last5"].append(mean_or_zero([x["goals_against"] for x in hist_a]))
    features["team_b_goals_against_avg_last5"].append(mean_or_zero([x["goals_against"] for x in hist_b]))

    features["team_a_weighted_goals_for_avg_last5"].append(mean_or_zero([x["weighted_goals_for"] for x in hist_a]))
    features["team_b_weighted_goals_for_avg_last5"].append(mean_or_zero([x["weighted_goals_for"] for x in hist_b]))

    features["team_a_weighted_goals_against_avg_last5"].append(mean_or_zero([x["weighted_goals_against"] for x in hist_a]))
    features["team_b_weighted_goals_against_avg_last5"].append(mean_or_zero([x["weighted_goals_against"] for x in hist_b]))

    features["team_a_avg_opponent_elo_last5"].append(mean_or_zero([x["opponent_elo"] for x in hist_a]))
    features["team_b_avg_opponent_elo_last5"].append(mean_or_zero([x["opponent_elo"] for x in hist_b]))

    features["team_a_rating_change_avg_last5"].append(mean_or_zero([x["rating_change"] for x in hist_a]))
    features["team_b_rating_change_avg_last5"].append(mean_or_zero([x["rating_change"] for x in hist_b]))

    features["team_a_matches_played_before"].append(len(hist_a_all))
    features["team_b_matches_played_before"].append(len(hist_b_all))

    features["team_a_days_since_last_match"].append((date - hist_a_all[-1]["date"]).days if hist_a_all else 999)
    features["team_b_days_since_last_match"].append((date - hist_b_all[-1]["date"]).days if hist_b_all else 999)

    features["team_a_win_streak"].append(current_win_streak(hist_a_all))
    features["team_b_win_streak"].append(current_win_streak(hist_b_all))

    opponent_weight_for_a = row["rating_b_before"] / ELO_BASE
    opponent_weight_for_b = row["rating_a_before"] / ELO_BASE

    team_history[team_a].append({
        "date": date,
        "points": row["result_a"],
        "goals_for": row["goals_a"],
        "goals_against": row["goals_b"],
        "weighted_goals_for": row["goals_a"] * opponent_weight_for_a,
        "weighted_goals_against": row["goals_b"] * (ELO_BASE / row["rating_b_before"]),
        "opponent_elo": row["rating_b_before"],
        "rating_change": row["rating_change_a"],
    })

    team_history[team_b].append({
        "date": date,
        "points": row["result_b"],
        "goals_for": row["goals_b"],
        "goals_against": row["goals_a"],
        "weighted_goals_for": row["goals_b"] * opponent_weight_for_b,
        "weighted_goals_against": row["goals_a"] * (ELO_BASE / row["rating_a_before"]),
        "opponent_elo": row["rating_a_before"],
        "rating_change": row["rating_change_b"],
    })

for col, values in features.items():
    df[col] = values

print("Rolling features created.")
print("ELO_BASE:", round(ELO_BASE, 2))

Rolling features created.
ELO_BASE: 1480.67


## 5. Tournament State Features

In [5]:
df["tournament_year"] = df["date"].dt.year
df["tournament_key"] = df["competition"].astype(str) + "_" + df["tournament_year"].astype(str)

tournament_states = {}

tournament_features = {
    "team_a_tournament_matches_played": [],
    "team_b_tournament_matches_played": [],
    "team_a_tournament_points": [],
    "team_b_tournament_points": [],
    "tournament_points_diff": [],
    "team_a_tournament_goal_diff": [],
    "team_b_tournament_goal_diff": [],
    "tournament_goal_diff_diff": [],
    "team_a_tournament_clean_sheets": [],
    "team_b_tournament_clean_sheets": [],
    "tournament_clean_sheets_diff": [],
}

def empty_state():
    return {
        "matches": 0,
        "points": 0,
        "goals_for": 0,
        "goals_against": 0,
        "goal_diff": 0,
        "clean_sheets": 0,
    }

for _, row in df.iterrows():
    key = row["tournament_key"]
    team_a = row["team_a"]
    team_b = row["team_b"]

    tournament_states.setdefault(key, {})
    tournament_states[key].setdefault(team_a, empty_state())
    tournament_states[key].setdefault(team_b, empty_state())

    state_a = tournament_states[key][team_a]
    state_b = tournament_states[key][team_b]

    tournament_features["team_a_tournament_matches_played"].append(state_a["matches"])
    tournament_features["team_b_tournament_matches_played"].append(state_b["matches"])
    tournament_features["team_a_tournament_points"].append(state_a["points"])
    tournament_features["team_b_tournament_points"].append(state_b["points"])
    tournament_features["tournament_points_diff"].append(state_a["points"] - state_b["points"])
    tournament_features["team_a_tournament_goal_diff"].append(state_a["goal_diff"])
    tournament_features["team_b_tournament_goal_diff"].append(state_b["goal_diff"])
    tournament_features["tournament_goal_diff_diff"].append(state_a["goal_diff"] - state_b["goal_diff"])
    tournament_features["team_a_tournament_clean_sheets"].append(state_a["clean_sheets"])
    tournament_features["team_b_tournament_clean_sheets"].append(state_b["clean_sheets"])
    tournament_features["tournament_clean_sheets_diff"].append(state_a["clean_sheets"] - state_b["clean_sheets"])

    state_a["matches"] += 1
    state_a["points"] += row["result_a"]
    state_a["goals_for"] += row["goals_a"]
    state_a["goals_against"] += row["goals_b"]
    state_a["goal_diff"] = state_a["goals_for"] - state_a["goals_against"]
    state_a["clean_sheets"] += int(row["goals_b"] == 0)

    state_b["matches"] += 1
    state_b["points"] += row["result_b"]
    state_b["goals_for"] += row["goals_b"]
    state_b["goals_against"] += row["goals_a"]
    state_b["goal_diff"] = state_b["goals_for"] - state_b["goals_against"]
    state_b["clean_sheets"] += int(row["goals_a"] == 0)

for col, values in tournament_features.items():
    df[col] = values

print("Tournament state features created.")
display(df[[
    "date", "competition", "team_a", "team_b",
    "team_a_tournament_matches_played",
    "team_b_tournament_matches_played",
    "team_a_tournament_points",
    "team_b_tournament_points",
    "tournament_points_diff"
]].head(12))

Tournament state features created.


,date,competition,team_a,team_b,team_a_tournament_matches_played,team_b_tournament_matches_played,team_a_tournament_points,team_b_tournament_points,tournament_points_diff
0,2004-01-01,Dudley Eve Memorial Trophy,Bermuda,Barbados,0,0,0,0,0
1,2004-01-01,Gulf Cup,Kuwait,Yemen,0,0,0,0,0
2,2004-01-01,Gulf Cup,Saudi Arabia,Bahrain,0,0,0,0,0
3,2004-01-03,Gulf Cup,Bahrain,Oman,1,0,0,0,0
4,2004-01-03,Gulf Cup,Qatar,United Arab Emirates,0,0,0,0,0
5,2004-01-04,Gulf Cup,Kuwait,Saudi Arabia,1,1,3,3,0
6,2004-01-05,Gulf Cup,Qatar,Yemen,1,1,1,0,1
7,2004-01-06,Gulf Cup,Saudi Arabia,Oman,2,1,4,0,4
8,2004-01-07,Gulf Cup,Bahrain,United Arab Emirates,2,1,3,1,2
9,2004-01-08,Friendly,Egypt,Rwanda,0,0,0,0,0


## 6. Competition Importance + Transfermarkt Market Values

In [6]:
def competition_weight(comp):
    comp = str(comp).lower()

    if comp == "world cup":
        return 5.0
    if (
        "euro" in comp
        or "copa america" in comp
        or "africa cup" in comp
        or "asian cup" in comp
        or "gold cup" in comp
    ):
        return 4.0
    if "nations league" in comp:
        return 2.5
    if "qualifier" in comp or "qualification" in comp:
        return 2.0
    if "friendly" in comp:
        return 0.5
    return 1.5

df["competition_weight"] = df["competition"].apply(competition_weight)

tm_path = PROCESSED_DIR / "transfermarkt_market_values_clean.csv"

if not tm_path.exists():
    raise FileNotFoundError(
        f"Missing Transfermarkt file: {tm_path}. "
        "Run 06_scrape_transfermarkt_market_values.ipynb first."
    )

tm = pd.read_csv(tm_path)

required_tm_cols = [
    "team_name_tm",
    "season_id",
    "squad_market_value_millions_eur",
    "avg_player_value_millions_eur",
    "market_value_relative_to_year_mean",
    "market_value_relative_to_year_median",
    "market_value_year_zscore",
    "log_market_value",
    "log_market_value_year_centered",
]

missing_tm = [c for c in required_tm_cols if c not in tm.columns]
if missing_tm:
    raise ValueError(f"Missing Transfermarkt columns: {missing_tm}")

tm = tm[required_tm_cols].copy()

name_map = {
    "USA": "United States",
    "Turkey": "Turkiye",
    "Czech Republic": "Czechia",
    "Cote d'Ivoire": "Ivory Coast",
    "DR Congo": "DR Congo",
    "Democratic Republic of the Congo": "DR Congo",
    "Korea Republic": "South Korea",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Cape Verde Islands": "Cape Verde",
    "Curacao": "Curaçao",
}

df["season_id"] = df["date"].dt.year
df["team_a_tm"] = df["team_a"].replace(name_map)
df["team_b_tm"] = df["team_b"].replace(name_map)

tm_a = tm.rename(columns={
    "team_name_tm": "team_a_tm",
    "squad_market_value_millions_eur": "market_value_a",
    "avg_player_value_millions_eur": "avg_player_value_a",
    "market_value_relative_to_year_mean": "market_value_rel_mean_a",
    "market_value_relative_to_year_median": "market_value_rel_median_a",
    "market_value_year_zscore": "market_value_zscore_a",
    "log_market_value": "log_market_value_a",
    "log_market_value_year_centered": "log_market_value_year_centered_a",
})

tm_b = tm.rename(columns={
    "team_name_tm": "team_b_tm",
    "squad_market_value_millions_eur": "market_value_b",
    "avg_player_value_millions_eur": "avg_player_value_b",
    "market_value_relative_to_year_mean": "market_value_rel_mean_b",
    "market_value_relative_to_year_median": "market_value_rel_median_b",
    "market_value_year_zscore": "market_value_zscore_b",
    "log_market_value": "log_market_value_b",
    "log_market_value_year_centered": "log_market_value_year_centered_b",
})

df = df.merge(tm_a, on=["team_a_tm", "season_id"], how="left")
df = df.merge(tm_b, on=["team_b_tm", "season_id"], how="left")

print("Market value coverage:")
print("team_a missing:", round(df["market_value_a"].isna().mean(), 3))
print("team_b missing:", round(df["market_value_b"].isna().mean(), 3))

market_cols = [
    "market_value_a",
    "market_value_b",
    "avg_player_value_a",
    "avg_player_value_b",
    "market_value_rel_mean_a",
    "market_value_rel_mean_b",
    "market_value_rel_median_a",
    "market_value_rel_median_b",
    "market_value_zscore_a",
    "market_value_zscore_b",
    "log_market_value_a",
    "log_market_value_b",
    "log_market_value_year_centered_a",
    "log_market_value_year_centered_b",
]

for col in market_cols:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

df["market_value_diff"] = df["market_value_a"] - df["market_value_b"]

df["market_value_ratio"] = (
    (df["market_value_a"] + 1) /
    (df["market_value_b"] + 1)
)

df["avg_player_value_diff"] = (
    df["avg_player_value_a"] -
    df["avg_player_value_b"]
)

df["market_value_rel_mean_diff"] = (
    df["market_value_rel_mean_a"] -
    df["market_value_rel_mean_b"]
)

df["market_value_rel_median_diff"] = (
    df["market_value_rel_median_a"] -
    df["market_value_rel_median_b"]
)

df["market_value_zscore_diff"] = (
    df["market_value_zscore_a"] -
    df["market_value_zscore_b"]
)

df["log_market_value_diff"] = (
    df["log_market_value_a"] -
    df["log_market_value_b"]
)

df["log_market_value_year_centered_diff"] = (
    df["log_market_value_year_centered_a"] -
    df["log_market_value_year_centered_b"]
)

Market value coverage:
team_a missing: 0.04
team_b missing: 0.048


## 7. Final Focused Feature Table

In [7]:
df["form_diff_last5"] = df["team_a_form_last5"] - df["team_b_form_last5"]

df["goals_for_diff_last5"] = (
    df["team_a_goals_for_avg_last5"] -
    df["team_b_goals_for_avg_last5"]
)

df["goals_against_diff_last5"] = (
    df["team_a_goals_against_avg_last5"] -
    df["team_b_goals_against_avg_last5"]
)

df["weighted_goals_for_diff_last5"] = (
    df["team_a_weighted_goals_for_avg_last5"] -
    df["team_b_weighted_goals_for_avg_last5"]
)

df["weighted_goals_against_diff_last5"] = (
    df["team_a_weighted_goals_against_avg_last5"] -
    df["team_b_weighted_goals_against_avg_last5"]
)

df["opponent_strength_diff_last5"] = (
    df["team_a_avg_opponent_elo_last5"] -
    df["team_b_avg_opponent_elo_last5"]
)

df["rating_change_diff_last5"] = (
    df["team_a_rating_change_avg_last5"] -
    df["team_b_rating_change_avg_last5"]
)

df["days_since_match_diff"] = (
    df["team_a_days_since_last_match"] -
    df["team_b_days_since_last_match"]
)

df["win_streak_diff"] = (
    df["team_a_win_streak"] -
    df["team_b_win_streak"]
)

FEATURE_COLS = [
    # Team strength
    "rating_a_before",
    "rating_b_before",
    "elo_diff",
    "rank_diff",

    # Transfermarkt market value
    "log_market_value_a",
    "log_market_value_b",
    "log_market_value_diff",

    "log_market_value_year_centered_a",
    "log_market_value_year_centered_b",
    "log_market_value_year_centered_diff",

    "market_value_rel_mean_a",
    "market_value_rel_mean_b",
    "market_value_rel_mean_diff",

    "market_value_zscore_diff",
    "avg_player_value_diff",

    # Recent performance
    "form_diff_last5",
    "weighted_goals_for_diff_last5",
    "weighted_goals_against_diff_last5",
    "opponent_strength_diff_last5",
    "rating_change_diff_last5",

    # Experience and schedule
    "team_a_matches_played_before",
    "team_b_matches_played_before",
    "team_a_days_since_last_match",
    "team_b_days_since_last_match",
    "days_since_match_diff",

    # Tournament state
    "team_a_tournament_matches_played",
    "team_b_tournament_matches_played",
    "tournament_points_diff",
    "tournament_goal_diff_diff",

    # Context
    "competition_weight",
    "is_home_adv",
]

TARGET_COLS = ["target_goals_a", "target_goals_b"]

metadata_cols = [
    "date",
    "team_a",
    "team_b",
    "competition",
    "location",
    "season_id",
    "tournament_year",
    "tournament_key",
]

model_df = df[
    metadata_cols
    + FEATURE_COLS
    + TARGET_COLS
    + ["target_goal_diff", "target_total_goals"]
].copy()

model_df["team_a_days_since_last_match"] = (
    model_df["team_a_days_since_last_match"].clip(0, 60)
)

model_df["team_b_days_since_last_match"] = (
    model_df["team_b_days_since_last_match"].clip(0, 60)
)

model_df["days_since_match_diff"] = (
    model_df["days_since_match_diff"].clip(-60, 60)
)

output_path = PROCESSED_DIR / "model_dataset.csv"
model_df.to_csv(output_path, index=False)

print("Final model dataset saved")
print("Shape:", model_df.shape)
print("Number of features:", len(FEATURE_COLS))
print("Saved to:", output_path)

display(model_df.head())

Final model dataset saved
Shape: (21539, 43)
Number of features: 31
Saved to: ../data/processed/model_dataset.csv


,date,team_a,team_b,competition,location,season_id,tournament_year,tournament_key,rating_a_before,rating_b_before,elo_diff,rank_diff,log_market_value_a,log_market_value_b,log_market_value_diff,log_market_value_year_centered_a,log_market_value_year_centered_b,log_market_value_year_centered_diff,market_value_rel_mean_a,market_value_rel_mean_b,market_value_rel_mean_diff,market_value_zscore_diff,avg_player_value_diff,form_diff_last5,weighted_goals_for_diff_last5,weighted_goals_against_diff_last5,opponent_strength_diff_last5,rating_change_diff_last5,team_a_matches_played_before,team_b_matches_played_before,team_a_days_since_last_match,team_b_days_since_last_match,days_since_match_diff,team_a_tournament_matches_played,team_b_tournament_matches_played,tournament_points_diff,tournament_goal_diff_diff,competition_weight,is_home_adv,target_goals_a,target_goals_b,target_goal_diff,target_total_goals
0,2004-01-01,Bermuda,Barbados,Dudley Eve Memorial Trophy,Bermuda,2004,2004,Dudley Eve Memorial Trophy_2004,1270,1369,-99,52,0.693147,0.0,0.693147,-0.408292,-1.101439,0.693147,0.016384,0.0,0.016384,0.009124,1.0,0.0,0.0,0.000000,0.0,0.0,0,0,60,60,0,0,0,0,0,1.5,1,0,4,-4,4
1,2004-01-01,Kuwait,Yemen,Gulf Cup,Kuwait,2004,2004,Gulf Cup_2004,1519,1179,340,-91,0.000000,0.0,0.000000,-1.101439,-1.101439,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0,0,60,60,0,0,0,0,0,1.5,1,4,0,4,4
2,2004-01-01,Saudi Arabia,Bahrain,Gulf Cup,Kuwait,2004,2004,Gulf Cup_2004,1662,1523,139,-43,0.000000,0.0,0.000000,-1.101439,-1.101439,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0,0,60,60,0,0,0,0,0,1.5,0,1,0,1,1
3,2004-01-03,Bahrain,Oman,Gulf Cup,Kuwait,2004,2004,Gulf Cup_2004,1511,1560,-49,-10,0.000000,0.0,0.000000,-1.101439,-1.101439,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.890897,1662.0,-12.0,1,0,2,60,-60,1,0,0,-1,1.5,0,1,0,1,1
4,2004-01-03,Qatar,United Arab Emirates,Gulf Cup,Kuwait,2004,2004,Gulf Cup_2004,1521,1455,66,-8,0.000000,0.0,0.000000,-1.101439,-1.101439,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0,0,60,60,0,0,0,0,0,1.5,0,0,0,0,0


## 8. Dataset Validation

In [8]:
missing_values = model_df[FEATURE_COLS + TARGET_COLS].isna().sum()
missing_values = missing_values[missing_values > 0]

if len(missing_values) > 0:
    display(missing_values)
    raise ValueError("Missing values found in model features or targets.")

non_numeric_features = [
    col for col in FEATURE_COLS
    if not pd.api.types.is_numeric_dtype(model_df[col])
]

if non_numeric_features:
    raise TypeError(f"Non-numeric feature columns found: {non_numeric_features}")

if (model_df[TARGET_COLS] < 0).any().any():
    raise ValueError("Negative goals found in target columns.")

print("Validation passed")
print("Rows:", len(model_df))
print("Features:", len(FEATURE_COLS))
print("Date range:", model_df["date"].min().date(), "to", model_df["date"].max().date())

Validation passed
Rows: 21539
Features: 31
Date range: 2004-01-01 to 2026-05-16


## 9. Poisson Score Conversion

In [9]:
def poisson_prob(lam, k):
    return (lam ** k) * exp(-lam) / factorial(k)

def most_likely_score(lambda_a, lambda_b, max_goals=6):
    lambda_a = min(max(float(lambda_a), 0.05), 4.0)
    lambda_b = min(max(float(lambda_b), 0.05), 4.0)

    best_score = None
    best_prob = -1

    for ga in range(max_goals + 1):
        for gb in range(max_goals + 1):
            prob = poisson_prob(lambda_a, ga) * poisson_prob(lambda_b, gb)

            if prob > best_prob:
                best_prob = prob
                best_score = (ga, gb)

    return best_score

## 10. Expanding-Window World Cup Backtest

In [10]:
model_df["is_world_cup"] = (
    model_df["competition"]
    .astype(str)
    .str.lower()
    .eq("world cup")
    .astype(int)
)

# Transfermarkt data starts in 2004, so evaluate from 2004 onward
model_df_eval = model_df[
    model_df["date"].dt.year >= 2004
].copy()

world_cup_years = sorted(
    model_df_eval.loc[
        model_df_eval["is_world_cup"] == 1,
        "date"
    ].dt.year.unique()
)

print("World Cup years:", world_cup_years)

results = []
all_predictions = []

for year in world_cup_years:
    wc_df = model_df_eval[
        (model_df_eval["is_world_cup"] == 1) &
        (model_df_eval["date"].dt.year == year)
    ].copy()

    wc_start = wc_df["date"].min()

    train_df = model_df_eval[
        (model_df_eval["date"] < wc_start) &
        (model_df_eval["rating_a_before"] >= 1420) &
        (model_df_eval["rating_b_before"] >= 1420)
    ].copy()

    test_df = wc_df.copy()

    X_train = train_df[FEATURE_COLS]
    y_train = train_df[TARGET_COLS]

    X_test = test_df[FEATURE_COLS]
    y_test = test_df[TARGET_COLS]

    model = MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=600,
            max_depth=10,
            min_samples_leaf=8,
            random_state=42,
            n_jobs=-1
        )
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)

    pred_scores = [
        most_likely_score(preds[i][0], preds[i][1])
        for i in range(len(preds))
    ]

    actual_scores = [
        (
            int(y_test.iloc[i]["target_goals_a"]),
            int(y_test.iloc[i]["target_goals_b"])
        )
        for i in range(len(y_test))
    ]

    exact_hits = [
        pred_scores[i] == actual_scores[i]
        for i in range(len(actual_scores))
    ]

    exact_acc = np.mean(exact_hits)

    results.append({
        "world_cup_year": int(year),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "rmse": rmse,
        "mae": mae,
        "exact_score_accuracy": exact_acc,
        "exact_score_accuracy_%": round(exact_acc * 100, 2),
    })

    pred_table = test_df[
        ["date", "team_a", "team_b", "target_goals_a", "target_goals_b"]
    ].copy()

    pred_table["pred_goals_a_reg"] = preds[:, 0]
    pred_table["pred_goals_b_reg"] = preds[:, 1]
    pred_table["pred_score"] = [f"{a}-{b}" for a, b in pred_scores]
    pred_table["actual_score"] = [f"{a}-{b}" for a, b in actual_scores]
    pred_table["exact_hit"] = exact_hits
    pred_table["world_cup_year"] = int(year)

    all_predictions.append(pred_table)

results_df = pd.DataFrame(results)
predictions_df = pd.concat(all_predictions, ignore_index=True)

display(results_df)
display(predictions_df.head(20))

World Cup years: [np.int32(2006), np.int32(2010), np.int32(2014), np.int32(2018), np.int32(2022)]


,world_cup_year,train_rows,test_rows,rmse,mae,exact_score_accuracy,exact_score_accuracy_%
0,2006,1217,64,0.939873,0.758093,0.218750,21.88
1,2010,3122,64,0.961083,0.761823,0.171875,17.19
2,2014,5088,64,1.052036,0.775803,0.093750,9.38
3,2018,6894,64,0.904066,0.710362,0.187500,18.75
4,2022,8691,64,0.956704,0.725693,0.296875,29.69


,date,team_a,team_b,target_goals_a,target_goals_b,pred_goals_a_reg,pred_goals_b_reg,pred_score,actual_score,exact_hit,world_cup_year
0,2006-06-09,Ecuador,Poland,2,0,2.799482,0.552006,2-0,2-0,True,2006
1,2006-06-09,Germany,Costa Rica,4,2,2.386070,0.350016,2-0,4-2,False,2006
2,2006-06-10,England,Paraguay,1,0,2.325199,0.585699,2-0,1-0,False,2006
3,2006-06-10,Sweden,Trinidad and Tobago,0,0,2.271913,0.406227,2-0,0-0,False,2006
4,2006-06-10,Argentina,Ivory Coast,2,1,2.186751,0.550124,2-0,2-1,False,2006
5,2006-06-11,Mexico,Iran,3,1,2.771280,0.610415,2-0,3-1,False,2006
6,2006-06-11,Netherlands,Serbia and Montenegro,1,0,2.080700,0.531920,2-0,1-0,False,2006
7,2006-06-11,Portugal,Angola,1,0,2.953696,0.286799,2-0,1-0,False,2006
8,2006-06-12,Australia,Japan,3,1,2.795767,0.302325,2-0,3-1,False,2006
9,2006-06-12,Czechia,United States,3,0,2.255147,0.623810,2-0,3-0,False,2006


## 11. Feature Importance

In [11]:
importances_a = model.estimators_[0].feature_importances_
importances_b = model.estimators_[1].feature_importances_

importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_goals_a": importances_a,
    "importance_goals_b": importances_b,
})

importance_df["avg_importance"] = (
    importance_df["importance_goals_a"] +
    importance_df["importance_goals_b"]
) / 2

importance_df = importance_df.sort_values(
    "avg_importance",
    ascending=False
).reset_index(drop=True)

display(importance_df)

,feature,importance_goals_a,importance_goals_b,avg_importance
0,rank_diff,0.464797,0.449716,0.457256
1,elo_diff,0.132485,0.126054,0.129270
2,rating_a_before,0.030321,0.093815,0.062068
3,rating_b_before,0.059111,0.026773,0.042942
4,opponent_strength_diff_last5,0.019762,0.020990,0.020376
5,weighted_goals_for_diff_last5,0.019905,0.020081,0.019993
6,weighted_goals_against_diff_last5,0.018119,0.018469,0.018294
7,avg_player_value_diff,0.022828,0.013533,0.018181
8,team_b_matches_played_before,0.015498,0.019422,0.017460
9,team_a_matches_played_before,0.014709,0.018676,0.016692


## 12. Save Evaluation Outputs

In [12]:
results_path = OUTPUTS_DIR / "world_cup_backtest_results.csv"
predictions_path = OUTPUTS_DIR / "world_cup_predictions.csv"
importance_path = OUTPUTS_DIR / "feature_importance.csv"

results_df.to_csv(results_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
importance_df.to_csv(importance_path, index=False)

print("Saved evaluation outputs:")
print("-", results_path)
print("-", predictions_path)
print("-", importance_path)

Saved evaluation outputs:
- ../outputs/evaluation/world_cup_backtest_results.csv
- ../outputs/evaluation/world_cup_predictions.csv
- ../outputs/evaluation/feature_importance.csv


## Summary

This notebook creates the final feature-engineering dataset for the World Cup score prediction model.

Main output:

- `data/processed/model_dataset.csv`

Evaluation outputs:

- `outputs/evaluation/world_cup_backtest_results.csv`
- `outputs/evaluation/world_cup_predictions.csv`
- `outputs/evaluation/feature_importance.csv`

The evaluation uses expanding-window World Cup backtesting:
each World Cup is predicted using only matches that occurred before that tournament.